# 서연 기존 모듈 1 × 공통 전처리 × 승호 튜닝 비교
공통 `src/`, `01`, `02`는 수정하지 않습니다. 전처리는 main의 착용 80%·수면 병합 90분·최소 42일이고, 서연 다섯 통계 모델·MOMENT와 승호 네 단계를 40·30·30 시간순 분할에서 비교합니다.

사용자 요청으로 이번 공개 범위는 **미주입 자료의 열 방법 비교**입니다. MOMENT 합성 추론과 열 방법 합성 비교는 미실시입니다. 기존 통계 아홉 방법 합성 자료는 로컬에 보존하되 여기서는 표시하지 않습니다. 이 노트북은 완료된 집계와 검증 기록을 읽기만 하며 추론·합성·파일 쓰기를 시작하지 않습니다. 저장된 노트북에 실행 출력이나 원자료를 넣지 않습니다.
보고서의 후속 해석은 기존 원문을 그대로 보존한 추가문이며 새로운 신경망 추론이 아닙니다. 기존 완료 기록은 변경하지 않고 원문·추가문·전체 보고서와 집계 근거의 해시를 별도로 확인합니다.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/data_loader.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
WHO = '조서연'
RUN = ROOT / 'results/seoyeon_tuning/comparison_20261015T161000Z/recovery_20261015T2358Z/run_view'
from experiments.seoyeon.comparison_safety import verified_outputs
from experiments.seoyeon.comparison_io import sha256
evaluation = verified_outputs(RUN, 'clean_evaluation.json', ['alarm_summary.csv', 'coverage.csv', 'pairwise.csv', 'clean_thresholds.json'])
publication = json.loads((RUN / 'clean_only_report_manifest.json').read_text())
if publication.get('status') != 'passed' or publication.get('scope') != 'clean_only':
    raise ValueError('미주입 자료 보고서 검증이 완료되지 않았습니다')
def verify_completion(run, report_hash):
    record = json.loads((run.parent / 'clean_only_completion.json').read_text())
    expected = dict(status='complete', scope='clean_only', clean_completed_claim=True,
                    full_comparison_completed_claim=False, synthetic_moment_jobs_completed=0)
    if any(record.get(key) != value for key, value in expected.items()):
        raise ValueError('미주입 범위 최종 검증이 완료되지 않았습니다')
    if record.get('report_sha256') != report_hash:
        raise ValueError('최종 검증 기록과 보고서 해시 불일치')
    outputs = record.get('outputs_sha256', {})
    if not outputs or any(sha256(run.parent / name) != digest for name, digest in outputs.items()):
        raise ValueError('최종 검증 산출물 해시 불일치')
    return record
def verify_report_extension(report, producer_hash, run):
    from hashlib import sha256 as hash_bytes
    content = report.read_bytes()
    current_hash = hash_bytes(content).hexdigest()
    if current_hash == producer_hash:
        return None
    sidecar = report.with_suffix('.appendix.json')
    if not sidecar.exists():
        raise ValueError('변경된 보고서의 추가 기록이 없습니다')
    record = json.loads(sidecar.read_text())
    if (not isinstance(record, dict) or type(record.get('version')) is not int
            or record['version'] != 1 or record.get('kind') != 'append_only_report_extension'
            or record.get('base_commit') != '94377776228f63cf4cd009b0a74b28af53808d12'):
        raise ValueError('보고서 추가 기록 형식 또는 기준 커밋 불일치')
    for key in ['original_report_sha256', 'appendix_sha256', 'current_report_sha256']:
        digest = record.get(key)
        if not isinstance(digest, str) or len(digest) != 64 or any(c not in '0123456789abcdef' for c in digest):
            raise ValueError('보고서 추가 기록 해시 형식 불일치')
    prefix_size, suffix_size = record.get('original_report_bytes'), record.get('appendix_bytes')
    if (type(prefix_size) is not int or type(suffix_size) is not int
            or prefix_size <= 0 or suffix_size <= 0 or prefix_size + suffix_size != len(content)):
        raise ValueError('보고서 원문 또는 추가문 바이트 수 불일치')
    if record['original_report_sha256'] != producer_hash:
        raise ValueError('기존 제작 기록과 보고서 원문 해시 불일치')
    if hash_bytes(content[:prefix_size]).hexdigest() != producer_hash:
        raise ValueError('기존 보고서 원문이 변경되었습니다')
    if hash_bytes(content[prefix_size:]).hexdigest() != record['appendix_sha256']:
        raise ValueError('보고서 추가문 해시 불일치')
    if current_hash != record['current_report_sha256']:
        raise ValueError('보고서 전체 해시 불일치')
    evidence = record.get('evidence_sha256')
    allowed = {'alarm_summary.csv', 'coverage.csv', 'pairwise.csv', 'clean_thresholds.json'}
    if not isinstance(evidence, dict) or set(evidence) != allowed:
        raise ValueError('추가문의 집계 근거 목록 불일치')
    for name, digest in evidence.items():
        if (not isinstance(digest, str) or len(digest) != 64
                or any(c not in '0123456789abcdef' for c in digest) or sha256(run / name) != digest):
            raise ValueError('추가문의 집계 근거 해시 불일치')
    return record
completion = verify_completion(RUN, publication['report_sha256'])
print('작성자:', WHO, '읽기 전용:', True, '검증 범위:', publication['scope'])


In [ ]:
display(pd.read_csv(RUN / 'preprocessing_counts.csv'))
print(json.loads((RUN / 'split.json').read_text()))
print('공통 보정 인일:', evaluation['calibration_rows'], '공통 시험 인일:', evaluation['test_rows'])
display(pd.read_csv(RUN / 'coverage.csv').query("rate_id == 'r05' and split == 'test'"))


## MOMENT 실제 추론과 복구 범위
고정 MOMENT-1-large의 세 특징×512달력일 당일 복원입니다. 현재 관측이 입력에 있으므로 미래예측이 아닙니다. 새 복구 프로세스의 신규 9,550인일과 같은 실험에서 여섯 입력·점수·모델을 검증한 재사용 78,336인일을 구분합니다. 복구의 신경망 재실행 대조는 8행이며 전행 재추론이 아닙니다. 부분 파일의 중단 전 해시는 없어 소급 무결성을 주장하지 않습니다.

In [ ]:
status = json.loads((RUN / 'moment_clean/execution.json').read_text())
if status.get('status') != 'complete': raise ValueError('MOMENT 미주입 추론 미완료')
print({k: status.get(k) for k in ['status','requested_persondays','completed_persondays','new_inferred_persondays','exact_six_field_cache_persondays']})
print('MOMENT 합성 및 열 방법 합성 비교: 미실시, 사용자 승인으로 이번 공개 범위에서 제외')


In [ ]:
alarms = pd.read_csv(RUN / 'alarm_summary.csv')
display(alarms.query("scope == 'common' and split == 'test'"))
pairs = pd.read_csv(RUN / 'pairwise.csv')
display(pairs.query("a == 'moment' or b == 'moment'"))
report = ROOT / 'experiments/seoyeon/COMPARISON.md'
extension = verify_report_extension(report, publication['report_sha256'], RUN)
if extension is not None: print('기존 원문 보존 및 추가 해석 검증 완료; 신규 신경망 추론 아님')
display(Markdown(report.read_text()))


## 해석과 재현의 한계
경보율·집중도·방법 간 일치는 실제 상태 정답이 아닙니다. 임상 정확도·오탐률·최적 모델을 검증하지 않았습니다. 동일 참가자의 관측일 수 차이와 전체기간 최소 42일 선정의 영향을 포함합니다.
새 클론의 기본 설치·CSV 세 개만으로 모든 재현이 끝나지 않습니다. 별도 원모델·기존 DOCX, MOMENT 실행환경·체크포인트·출처 기록, 이전 승호 원설정 재현에는 비공개 보관 코드와 Git 객체가 필요합니다. 이 노트북에는 이미 생성된 로컬 결과 경로가 필요합니다. 자세한 의존성과 미검증 범위는 보고서 8절에 있습니다.